In [39]:
import pvlib
import pypsa
import matplotlib.pyplot as plt
import xarray as xr
import pandas as pd
import numpy as np


In [41]:
# Load file names and prepare a list to collect results
import glob
liste = []
pdc0 = 1000

# Compute PV capacity factor from surface solar radiation (ssr) in a dataset
def calculate_cf_factor(ds: xr, albedo=0.1):

    # Adjust SSR for surface albedo (convert reflected to incident by dividing by (1 - albedo))
    ds['ssr'] = ds['ssr'] / (1-albedo)                 # albedo factor --> yes / no?

    # Convert accumulated radiation to step-wise values along the 'step' dimension
    ds = ds.diff('step')  # J/m²                    # TIGGE has accumulated values, therefore the difference 
                                                    # is the actual radiation in J / m^2

    # Convert J/m² per 6-hour step to Wh/m², clean NaNs, and clip to non-negative
    ds = (ds/(3600*6)).fillna(0).clip(min=0)            # get Wh / m^2
    ds = ds.where(ds > 1, 0)

    # Convert irradiance to capacity factor using module DC capacity (pdc0) and a scaling factor
    ds['cf'] = 0.9 * ds['ssr'] / (pdc0)         

    # Return only the capacity-factor variable
    return ds['cf']


# Build PyPSA-style export: one DataFrame per grid point/asset with a 3-hourly index
def get_pypsa_export_structure(data: xr):

    cf = []

    # Iterate over assets indexed by 'number'
    for i in range(0, data.number[-1].values):
        # Define the target time index (adjust end as needed)
        index = pd.date_range(
            start="2019-01-01 00:00",
            # CHANGE END HERE
            end="2020-01-02 00:00",
            freq="3h"
        )

        # Prepare an empty column for this asset
        df = pd.DataFrame(index=index)
        df[f'cf_solar_{i+1}'] = np.NaN

        # Fill values from data chunks by aligning their 'valid_time' to the index
        for data_chunk in data:
            dt = data_chunk['valid_time'].values
            dt_str = pd.to_datetime(dt).strftime("%Y-%m-%d %H:%M:%S")  
            df.loc[dt_str, f'cf_solar_{i+1}'] = data_chunk[i].values

        # Interpolate only during typical daylight hours; leave nighttime as-is
        #df[f'cf_solar_{i+1}'] = df[f'cf_solar_{i+1}'].interpolate(method="linear")
        mask = (df.index.time >= pd.to_datetime("08:00").time()) & (df.index.time <= pd.to_datetime("19:00").time())
        df.loc[mask, f'cf_solar_{i+1}'] = df.loc[mask, f'cf_solar_{i+1}'].interpolate(method='linear')
        
        # Shift the series by one step earlier and fill any remaining gaps with zeros
        df = df.shift(-1)
        df = df.fillna(0)
        cf.append(df)

    # Return the list of DataFrames (one per asset)
    return cf


# End-to-end pipeline: open GRIB files, compute PV capacity factor, and format for PyPSA export
def get_pv_capacity_factor():

    # Iterate through all ECMWF GRIB files containing radiation variables
    for file_name in glob.glob("/Users/mick/Documents/GitHub/masterthesis-mick/Wetterdaten/ECWMF_Data/*.grb"):
        solar_data = xr.open_dataset(
                file_name,
                engine="cfgrib",
                backend_kwargs={
                    "filter_by_keys": {
                        "paramId": [167, 176],
                    }
                }
            )
        # Optionally select a single point; here we use the spatial mean instead
        #data_chunk = solar_data.loc[{'latitude': 53.0, 'longitude': 10.0}]
        data_chunk = solar_data.mean(dim=("latitude", "longitude"))

        # Compute capacity factor from SSR and prepare a time dimension label
        cf = calculate_cf_factor(data_chunk)
        cf.expand_dims(time=[str(data_chunk['time'].values)[:10]])

        # Collect into the list for later concatenation
        liste.append(cf)

    # Concatenate all daily capacity-factor slices along 'time' and sort chronologically
    data = xr.concat(liste, dim='time')
    data = data.sortby("time")

    # Reshape into PyPSA export structure (list of DataFrames)
    data = get_pypsa_export_structure(data)

    # Return the final list of DataFrames
    return data


In [42]:
cf_list = get_pv_capacity_factor()

for i in range(0,50):
    cf_list[i] = cf_list[i].loc[:'2019-12-31 22:00']

In [3]:
## read
#cf_list = pd.read_pickle('/Users/mick/Documents/GitHub/masterthesis-mick/Wetterdaten/cappacity_factors_prepared/cf_solar.pkl')

## write
#pd.to_pickle(cf_list, "/Users/mick/Documents/GitHub/masterthesis-mick/Wetterdaten/cappacity_factors_prepared/cf_solar.pkl")